# Task 4: Reinforcement Learning from Human Feedback (RLHF)

**Goal:** Fine-tune the pretrained Task 3 Transformer using human preference scores.

**Mathematical Model:**
- Optimization Objective: $\max_\theta J(\theta) = \mathbb{E}[r(X_{gen})]$
- Policy Gradient (REINFORCE): $\nabla_\theta J(\theta) = \mathbb{E}[r \nabla_\theta \log p_\theta(X)]$
- KL Penalty (Guide Fix): $J'(\theta) = \mathbb{E}[r] - \lambda D_{KL}(p_\theta || p_{\theta_0})$ to prevent Reward Hacking.

**Deliverables:** Human survey data integration, reward scoring function, RL tuning loop, 10 final tuned samples.

In [1]:
import torch, os, math
import numpy as np
from torch import nn, optim
import torch.nn.functional as F
from miditok import REMI, TokenizerConfig

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

config = TokenizerConfig(num_velocities=32, use_chords=False, use_programs=False)
tokenizer = REMI(config)
VOCAB_SIZE = tokenizer.vocab_size
PAD_TOKEN = tokenizer['PAD_None']

### 1. Load Pretrained Task 3 Model & Reference Model
We need the active model (`model`) to train, and a frozen reference model (`ref_model`) to compute the KL penalty and prevent the music from devolving into noise just to exploit the reward function.

In [5]:
# Re-instantiate the GPT architecture from Task 3
class GPTMusic(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=8, num_layers=4):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(1024, d_model)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4, batch_first=True, dropout=0.1)
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        positions = torch.arange(0, seq_len, device=x.device).unsqueeze(0)
        x_emb = self.token_emb(x) + self.pos_emb(positions)
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len, seq_len, device=x.device)
        out = self.transformer(x_emb, mask=mask, is_causal=True)
        return self.fc(out)

model = GPTMusic(VOCAB_SIZE).to(device)
ref_model = GPTMusic(VOCAB_SIZE).to(device)

# In practice, load your saved Task 3 weights here:
model_path = 'models/saved/transformer.pth'
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    ref_model.load_state_dict(torch.load(model_path, map_location=device))
    print("Loaded pretrained Task 3 weights.")
else:
    print("Warning: Pretrained Task 3 weights not found. Using random init.")

# Freeze the reference model
ref_model.eval()
for param in ref_model.parameters():
    param.requires_grad = False

### 2. Reward Function (Mock Setup)
In your report, you must do a human survey. For automation, you can train a tiny neural network (Reward Model) on those 1-5 human scores to predict rewards automatically during the RL loop. Below is a mock reward function.

In [3]:
def get_mock_human_reward(sequences):
    """
    Placeholder for your actual trained Reward Model or scoring algorithm.
    For example, returning higher rewards for a good ratio of unique sequences,
    or evaluating evaluating harmony.
    """
    # For structural implementation, return random scores centered around 3.0 (Scale: 1-5)
    # Replace this with your actual metric/survey model inferences.
    return torch.normal(mean=3.0, std=0.5, size=(sequences.size(0),)).to(device)

### 3. Policy Gradient Update Loop (REINFORCE with Baseline & KL Penalty)

In [6]:
# RL requires a very small learning rate to avoid destructive updates
optimizer = optim.Adam(model.parameters(), lr=5e-5)
KL_BETA = 0.1 # Penalty multiplier for deviating too far from Task 3 structure
RL_STEPS = 50
BATCH_SIZE = 8
SEQ_LEN = 128

print("Starting RLHF Tuning...")
model.train()

for step in range(1, RL_STEPS + 1):
    # 1. Generate dummy combinations (In full implementation, use your dataset's initial seeds)
    # Using random tokens for structural demonstration as actual generation requires autoregressive iteration
    dummy_inputs = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN)).to(device)
    
    optimizer.zero_grad()
    
    # 2. Get Active Model Logits
    logits = model(dummy_inputs)
    log_probs = F.log_softmax(logits, dim=-1)
    
    # Extract log probabilities of the actual chosen tokens
    # Shift targets by 1 for autoregressive next-token prob
    target_tokens = dummy_inputs[:, 1:].unsqueeze(-1)
    selected_log_probs = log_probs[:, :-1, :].gather(2, target_tokens).squeeze(-1).sum(dim=1)

    # 3. Get Reference Model Logits (No Grad) for KL Divergence
    with torch.no_grad():
        ref_logits = ref_model(dummy_inputs)
        ref_log_probs = F.log_softmax(ref_logits, dim=-1)
        ref_selected_log_probs = ref_log_probs[:, :-1, :].gather(2, target_tokens).squeeze(-1).sum(dim=1)
        
    # 4. Calculate RLHF Penalty (KL Divergence)
    kl_div = selected_log_probs - ref_selected_log_probs
    
    # 5. Acquire Human Rewards & Normalize (High Variance Fix)
    rewards = get_mock_human_reward(dummy_inputs)
    normalized_rewards = (rewards - rewards.mean()) / (rewards.std() + 1e-8)
    
    # 6. Expectation Objective with KL Penalty (Ascent translates to minimizing negative)
    # Loss = - E[ R * log_p(x) - Beta * KL(p || p_ref) ]
    rl_loss = - (normalized_rewards * selected_log_probs).mean()
    kl_loss = KL_BETA * kl_div.mean()
    
    total_loss = rl_loss + kl_loss
    
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    if step % 10 == 0:
        print(f"Step {step}/{RL_STEPS} | Total Loss: {total_loss.item():.4f} | RL Obj: {-rl_loss.item():.4f} | KL Pen: {kl_loss.item():.4f}")

os.makedirs('models/saved', exist_ok=True)
torch.save(model.state_dict(), 'models/saved/transformer_rlhf.pth')
print("RLHF Tuning Complete.")

Starting RLHF Tuning...
Step 10/50 | Total Loss: -3.9346 | RL Obj: 4.0089 | KL Pen: 0.0743
Step 20/50 | Total Loss: 0.8061 | RL Obj: -0.6687 | KL Pen: 0.1373
Step 10/50 | Total Loss: -3.9346 | RL Obj: 4.0089 | KL Pen: 0.0743
Step 20/50 | Total Loss: 0.8061 | RL Obj: -0.6687 | KL Pen: 0.1373
Step 30/50 | Total Loss: -0.2354 | RL Obj: -0.2167 | KL Pen: -0.4522
Step 40/50 | Total Loss: 0.7933 | RL Obj: -1.0085 | KL Pen: -0.2153
Step 30/50 | Total Loss: -0.2354 | RL Obj: -0.2167 | KL Pen: -0.4522
Step 40/50 | Total Loss: 0.7933 | RL Obj: -1.0085 | KL Pen: -0.2153
Step 50/50 | Total Loss: -0.9045 | RL Obj: 0.0955 | KL Pen: -0.8090
RLHF Tuning Complete.
Step 50/50 | Total Loss: -0.9045 | RL Obj: 0.0955 | KL Pen: -0.8090
RLHF Tuning Complete.
